## 16_b_bootstrap
  Region bootstrap (1000 draws, checkpointed in chunks) of b for CZ and PL; confidence
  intervals, capital-excluded refits, paired b - beta differences and equivalence margins.
  Output: data/processed/16_b_bootstrap/. Environment: nuts3-mobility.

In [ ]:
import os
import sys
from importlib.util import find_spec
from pathlib import Path

# PROJ data fix: must run BEFORE geopandas / pyproj / skmob are imported
def find_proj_dir():
    """Return the first directory that contains proj.db (no pyproj import needed)."""
    candidates = [Path(sys.prefix) / "share" / "proj"]  # conda environment
    spec = find_spec("pyproj")
    if spec is not None and spec.submodule_search_locations:
        pyproj_dir = Path(list(spec.submodule_search_locations)[0])
        candidates.append(pyproj_dir / "proj_dir" / "share" / "proj")
    for cand in candidates:
        if (cand / "proj.db").exists():
            return cand
    return None

proj_dir = find_proj_dir()
if proj_dir is not None:
    os.environ["PROJ_LIB"] = str(proj_dir)
    os.environ["PROJ_DATA"] = str(proj_dir)
    print("PROJ data: proj.db found, environment variables set")
else:
    print("PROJ data: proj.db NOT found in the expected places")

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray

sys.path.append(str(Path.cwd().parent))  # notebook is in notebooks/
from src import config

# ----------------------------- CONFIG -----------------------------
YEAR = 2023                                   # same year as notebooks 04, 05, 05b
COL_ID, COL_POP = "nuts3_id", "population"
COL_COUNTRY, COL_YEAR = "country_code", "year"
CRS_GEO = "EPSG:4326"
NUTS3_TABLE = config.PROCESSED_DIR / "nuts3_master_table.parquet"

# Folders of the original fits and the file names saved there
SOURCE_DIRS = {
    "CZ": config.PROCESSED_DIR / "05_gravity_cz",
    "PL": config.PROCESSED_DIR / "05b_gravity_pl",
}
OD_FILE, PARAMS_FILE = "od_observed.csv", "gravity_params.csv"

# Fit sample rules per variant; the variant name is the tag of all output files.
# main_fit = True: the rules are those of the original fit (05 / 05b)
VARIANTS = {
    "CZ_nodiag": {"country": "CZ", "include_diagonal": False, "drop_zero_pairs": False, "main_fit": False},
    "PL_main":   {"country": "PL", "include_diagonal": False, "drop_zero_pairs": True,  "main_fit": True},
}

# Output folder of this notebook (checkpoints and result tables)
OUT_DIR = config.PROCESSED_DIR / "16_b_bootstrap"
# -------------------------------------------------------------------

COUNTRIES = list(dict.fromkeys(v["country"] for v in VARIANTS.values()))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Master table: geometry is stored as WKB bytes; build it element by element
nuts3 = pd.read_parquet(NUTS3_TABLE)
geom_arr = np.empty(len(nuts3), dtype=object)
for k, wkb_bytes in enumerate(nuts3["geometry"]):
    geom_arr[k] = shapely.wkb.loads(wkb_bytes) if wkb_bytes is not None else None
nuts3 = nuts3.drop(columns="geometry")
nuts3 = gpd.GeoDataFrame(nuts3, geometry=GeometryArray(geom_arr, crs=CRS_GEO))

regions, od_obs, main_params = {}, {}, {}
for c in COUNTRIES:
    # Regions of the country in the same order as in the original notebooks
    reg = nuts3[(nuts3[COL_COUNTRY] == c) & (nuts3[COL_YEAR] == YEAR)]
    reg = reg[[COL_ID, "geometry", COL_POP]].reset_index(drop=True)

    # Observed OD matrix (rows = origin, columns = destination), aligned to the region order
    od = pd.read_csv(SOURCE_DIRS[c] / OD_FILE, index_col=0)
    assert set(od.index) == set(od.columns) == set(reg[COL_ID]), f"{c}: region ids differ"
    od = od.loc[reg[COL_ID], reg[COL_ID]]

    params = pd.read_csv(SOURCE_DIRS[c] / PARAMS_FILE)
    regions[c], od_obs[c], main_params[c] = reg, od, params

    print(f"{c}: regions = {len(reg)} | missing population = {int(reg[COL_POP].isna().sum())}"
          f" | commuters in matrix = {int(od.to_numpy().sum())}")
    print(f"{c}: main destination_exp =",
          params.set_index("model")["destination_exp"].round(4).to_dict())

# Size of the fit sample of each variant under its rules
for tag, rule in VARIANTS.items():
    od = od_obs[rule["country"]]
    mask = np.ones(od.shape, dtype=bool)
    if not rule["include_diagonal"]:
        np.fill_diagonal(mask, False)
    if rule["drop_zero_pairs"]:
        mask &= od.to_numpy() > 0
    params = main_params[rule["country"]]
    check = (f" | n_fit_pairs in {PARAMS_FILE} = {int(params['n_fit_pairs'].iloc[0])}"
             if rule["main_fit"] and "n_fit_pairs" in params.columns else "")
    print(f"{tag}: pairs in fit sample = {int(mask.sum())}{check}")

In [ ]:
import os
import warnings

import skmob
from skmob.models.gravity import Gravity

# ----------------------------- CONFIG -----------------------------
TILE_ID = "tile_ID"
COL_FLOW_ORIGIN, COL_FLOW_DEST, COL_FLOW = "origin", "destination", "flow"
GRAVITY_TYPE = "singly constrained"
GRAVITY_VARIANTS = {"gravity_power": "power_law", "gravity_exp": "exponential"}
REFIT_CHECK_CSV = OUT_DIR / "refit_check.csv"
# -------------------------------------------------------------------


def build_fdf(country, draw, rule):
    """Build the skmob fit sample for a draw of regions (positions, repeats allowed)."""
    reg = regions[country]
    od = od_obs[country].to_numpy()
    draw = np.asarray(draw)
    orig_ids = reg[COL_ID].to_numpy()[draw]
    slot_ids = np.array([f"{rid}#{k}" for k, rid in enumerate(orig_ids)])

    # Pair rules are applied on the ORIGINAL region ids (copies of one region count as diagonal)
    flows = od[np.ix_(draw, draw)]
    mask = np.ones(flows.shape, dtype=bool)
    if not rule["include_diagonal"]:
        mask &= orig_ids[:, None] != orig_ids[None, :]
    if rule["drop_zero_pairs"]:
        mask &= flows > 0
    o_pos, d_pos = np.nonzero(mask)

    flows_long = pd.DataFrame({
        COL_FLOW_ORIGIN: slot_ids[o_pos],
        COL_FLOW_DEST: slot_ids[d_pos],
        COL_FLOW: flows[o_pos, d_pos],
    })

    # Tessellation keeps the geometry column of the region table (one row per slot)
    tess = reg.iloc[draw].reset_index(drop=True).rename(columns={COL_ID: TILE_ID})
    tess[TILE_ID] = slot_ids

    return skmob.FlowDataFrame(
        flows_long,
        origin=COL_FLOW_ORIGIN,
        destination=COL_FLOW_DEST,
        flow=COL_FLOW,
        tessellation=tess,
        tile_id=TILE_ID,
    )


def fit_b(fdf):
    """Fit both gravity variants; failures are recorded by class name, not raised."""
    out = {}
    for name, deterrence in GRAVITY_VARIANTS.items():
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                model = Gravity(gravity_type=GRAVITY_TYPE, deterrence_func_type=deterrence)
                model.fit(fdf, relevance_column=COL_POP)
                b, err = model.destination_exp, ""
            except Exception as e:
                b, err = np.nan, type(e).__name__
        out[name] = {"b": b, "error_class": err,
                     "warnings": ";".join(sorted({w.category.__name__ for w in caught}))}
    return out


# Refit on the full sample (every region exactly once) and compare with the saved b
rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    fdf_full = build_fdf(c, np.arange(len(regions[c])), rule)
    res = fit_b(fdf_full)
    saved = main_params[c].set_index("model")["destination_exp"] if rule["main_fit"] else None
    for model_name, r in res.items():
        b_saved = saved[model_name] if saved is not None else np.nan
        rows.append({"variant": tag, "model": model_name, "n_pairs": len(fdf_full),
                     "b_refit": r["b"], "b_saved": b_saved, "diff": r["b"] - b_saved,
                     "error_class": r["error_class"], "warnings": r["warnings"]})

refit_check = pd.DataFrame(rows)
refit_check.to_csv(REFIT_CHECK_CSV, index=False)
print(refit_check.round(6).to_string(index=False))
print("Saved:", os.path.relpath(REFIT_CHECK_CSV))

In [ ]:
import time

# ----------------------------- CONFIG -----------------------------
N_BOOT = 1000           # bootstrap draws per variant
CHUNK_SIZE = 20         # draws per checkpoint file
SEED_BASE = 0           # draw i uses seed SEED_BASE + i (same draws for all variants of a country)
BOOT_DIR = OUT_DIR / "boot_chunks"
# -------------------------------------------------------------------

BOOT_DIR.mkdir(parents=True, exist_ok=True)

for tag, rule in VARIANTS.items():
    c = rule["country"]
    n_reg = len(regions[c])
    for start in range(0, N_BOOT, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, N_BOOT)
        chunk_csv = BOOT_DIR / f"boot_{tag}_{start:04d}.csv"
        if chunk_csv.exists():
            continue                      # resume: this chunk is already done

        t0 = time.time()
        rows = []
        for i in range(start, stop):
            rng = np.random.default_rng(SEED_BASE + i)
            draw = rng.integers(0, n_reg, size=n_reg)   # regions drawn with replacement

            # A failed draw is recorded by class name, not raised
            try:
                fdf = build_fdf(c, draw, rule)
                n_pairs = len(fdf)
                res = fit_b(fdf)
            except Exception as e:
                n_pairs = np.nan
                res = {m: {"b": np.nan, "error_class": type(e).__name__, "warnings": ""}
                       for m in GRAVITY_VARIANTS}

            for model_name, r in res.items():
                rows.append({"variant": tag, "draw": i, "model": model_name,
                             "n_unique_regions": len(set(draw.tolist())), "n_pairs": n_pairs,
                             "b": r["b"], "error_class": r["error_class"],
                             "warnings": r["warnings"]})

        pd.DataFrame(rows).to_csv(chunk_csv, index=False)
        print(f"{tag}: draws {start}-{stop - 1} done in {time.time() - t0:.1f} s")

n_files = sum(len(list(BOOT_DIR.glob(f"boot_{tag}_*.csv"))) for tag in VARIANTS)
print("Chunk files:", n_files, "| expected:", len(VARIANTS) * -(-N_BOOT // CHUNK_SIZE))

In [ ]:
# ----------------------------- CONFIG -----------------------------
BOOT_ALL_CSV = OUT_DIR / "boot_all.csv"
BOOT_SUMMARY_CSV = OUT_DIR / "boot_summary.csv"
# -------------------------------------------------------------------

# Combine all chunk files into one table
boot = pd.concat([pd.read_csv(p) for tag in VARIANTS
                  for p in sorted(BOOT_DIR.glob(f"boot_{tag}_*.csv"))], ignore_index=True)
boot["error_class"] = boot["error_class"].fillna("")
boot.to_csv(BOOT_ALL_CSV, index=False)

# Summary per variant and model: empty b values and the range of b
summary = (boot.groupby(["variant", "model"])
           .agg(n_draws=("draw", "count"),
                n_nan=("b", lambda s: int(s.isna().sum())),
                n_error=("error_class", lambda s: int((s != "").sum())),
                b_min=("b", "min"), b_median=("b", "median"), b_max=("b", "max"))
           .reset_index())
summary.to_csv(BOOT_SUMMARY_CSV, index=False)

print(summary.round(4).to_string(index=False))
print()
# Error classes that occurred (empty if none)
errors = (boot.loc[boot["error_class"] != "", ["variant", "model", "error_class"]]
          .value_counts().rename("count").reset_index())
print(errors.to_string(index=False) if len(errors) else "No error classes recorded")
print("Saved:", os.path.relpath(BOOT_ALL_CSV), "|", os.path.relpath(BOOT_SUMMARY_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
CI_LEVEL = 0.95
CI_CSV = OUT_DIR / "b_bootstrap_ci.csv"
# -------------------------------------------------------------------

boot = pd.read_csv(BOOT_ALL_CSV)
q_lo, q_hi = (1 - CI_LEVEL) / 2, 1 - (1 - CI_LEVEL) / 2

rows = []
for (tag, model_name), g in boot.groupby(["variant", "model"]):
    b_valid = g["b"].dropna()                      # failed draws have an empty b
    b_main = (main_params[VARIANTS[tag]["country"]]
              .set_index("model")["destination_exp"][model_name])
    b_lo, b_hi = b_valid.quantile([q_lo, q_hi])
    rows.append({
        "variant": tag, "model": model_name,
        "n_draws": len(g), "n_valid": len(b_valid),
        "share_failed": 1 - len(b_valid) / len(g),
        "b_main": b_main, "b_lo": b_lo, "b_median": b_valid.median(), "b_hi": b_hi,
        "b_std": b_valid.std(ddof=1),
        "main_in_ci": bool(b_lo <= b_main <= b_hi),
    })

ci_table = pd.DataFrame(rows)
ci_table.to_csv(CI_CSV, index=False)

print(f"{int(CI_LEVEL * 100)}% percentile interval of b (valid draws only)")
print(ci_table.round(4).to_string(index=False))
print("Saved:", os.path.relpath(CI_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
FAIL_VARIANT = "PL_main"
N_BINS = 4                                        # groups by number of unique regions
FAIL_CSV = OUT_DIR / "pl_failed_draws_summary.csv"
# -------------------------------------------------------------------

fb = boot[boot["variant"] == FAIL_VARIANT].copy()
fb["failed"] = fb["error_class"].fillna("") != ""

# 1) Failed vs successful draws: size of the sample
size_summary = (fb.groupby(["model", "failed"])
                .agg(n=("draw", "count"),
                     uniq_min=("n_unique_regions", "min"),
                     uniq_median=("n_unique_regions", "median"),
                     uniq_max=("n_unique_regions", "max"),
                     pairs_min=("n_pairs", "min"),
                     pairs_median=("n_pairs", "median"),
                     pairs_max=("n_pairs", "max"))
                .reset_index())

# 2) Share of failed draws by group of unique-region count
fb["uniq_group"] = pd.qcut(fb["n_unique_regions"], q=N_BINS, duplicates="drop").astype(str)
share_by_group = (fb.groupby(["model", "uniq_group"])["failed"]
                  .agg(n_draws="count", n_failed="sum", share_failed="mean")
                  .reset_index())

# 3) Are the same draws failing in both models?
failed_sets = {m: set(g.loc[g["failed"], "draw"]) for m, g in fb.groupby("model")}
n_both = len(set.intersection(*failed_sets.values()))

size_summary.to_csv(FAIL_CSV, index=False)

print(size_summary.round(1).to_string(index=False))
print()
print(share_by_group.round(3).to_string(index=False))
print()
print("Failed draws per model:", {m: len(s) for m, s in failed_sets.items()},
      "| failed in both models:", n_both)
print("Saved:", os.path.relpath(FAIL_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
BETA_CSV = config.PROCESSED_DIR / "15_b_estimation" / "beta_values.csv"
BETA_SAMPLE = "full"                                    # all regions, as in notebook 14
COL_BETA_GROUP, COL_BETA_SAMPLE, COL_BETA_Y = "group", "sample", "y_variable"
COL_BETA, COL_BETA_LO, COL_BETA_HI = "beta", "beta_ci_low_hc3", "beta_ci_high_hc3"
COMPARE_CSV = OUT_DIR / "b_ci_vs_beta_ci.csv"
# -------------------------------------------------------------------

beta = pd.read_csv(BETA_CSV)
beta = beta[beta[COL_BETA_SAMPLE] == BETA_SAMPLE]

# b intervals from the bootstrap (one row per variant and model); add the country
b_ci = pd.read_csv(CI_CSV)
b_ci["country"] = b_ci["variant"].map(lambda tag: VARIANTS[tag]["country"])

cmp_table = b_ci.merge(
    beta[[COL_BETA_GROUP, COL_BETA_Y, COL_BETA, COL_BETA_LO, COL_BETA_HI]],
    left_on="country", right_on=COL_BETA_GROUP,
)
cmp_table["diff_b_minus_beta"] = cmp_table["b_main"] - cmp_table[COL_BETA]
cmp_table["ci_overlap"] = ((cmp_table["b_lo"] <= cmp_table[COL_BETA_HI])
                           & (cmp_table[COL_BETA_LO] <= cmp_table["b_hi"]))
cmp_table["beta_in_b_ci"] = cmp_table[COL_BETA].between(cmp_table["b_lo"], cmp_table["b_hi"])

cols = ["country", "model", COL_BETA_Y, "b_main", "b_lo", "b_hi",
        COL_BETA, COL_BETA_LO, COL_BETA_HI, "diff_b_minus_beta", "ci_overlap", "beta_in_b_ci"]
cmp_table = cmp_table[cols].sort_values(["country", "model", COL_BETA_Y]).reset_index(drop=True)
cmp_table.to_csv(COMPARE_CSV, index=False)

print(cmp_table.round(3).to_string(index=False))
print()
print(cmp_table.groupby(["country", "model"])[["ci_overlap", "beta_in_b_ci"]].sum().to_string())
print("Saved:", os.path.relpath(COMPARE_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
CZ_VARIANT = "CZ_nodiag"
LOW_B_THRESHOLD = 0.5                       # b below this value counts as "low"
UNIQUE_CSV = OUT_DIR / "cz_b_by_unique_regions.csv"
# -------------------------------------------------------------------

cz_boot = boot[boot["variant"] == CZ_VARIANT]

by_unique = (cz_boot.groupby(["model", "n_unique_regions"])["b"]
             .agg(n_draws="count", b_min="min", b_median="median", b_max="max",
                  share_low=lambda s: (s < LOW_B_THRESHOLD).mean(),
                  share_negative=lambda s: (s < 0).mean())
             .reset_index())
by_unique.to_csv(UNIQUE_CSV, index=False)

print(f"low b = b < {LOW_B_THRESHOLD}")
print(by_unique.round(3).to_string(index=False))
print("Saved:", os.path.relpath(UNIQUE_CSV))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ----------------------------- CONFIG -----------------------------
FIG_DIR = Path("../figures") / "b_bootstrap"
FIG_FILE = "b_bootstrap_hist.png"
FIG_DPI = 150
FIG_WIDTH, BLOCK_HEIGHT = 11, 4.6              # figure width and height of one country block
HEIGHT_RATIOS = (3, 2)                         # histogram row : beta row
BLOCK_GAP, ROW_GAP, COL_GAP = 0.45, 0.12, 0.12 # spacing between blocks, rows inside a block, columns
BOTTOM_MARGIN = 0.12                           # room for the legend under the last x label
N_HIST_BINS = 40
X_PAD = 0.05                                   # padding of the x range (fraction of the range)
MODEL_LABELS = {"gravity_power": "Power-law deterrence", "gravity_exp": "Exponential deterrence"}
X_LABEL, Y_LABEL = "exponent (b or beta)", "number of draws"
CI_COLOR, HIST_COLOR = "lightgray", "steelblue"
BETA_COLOR, NO_OVERLAP_COLOR = "black", "tab:red"
# -------------------------------------------------------------------

FIG_DIR.mkdir(parents=True, exist_ok=True)

tags = list(VARIANTS)
models = list(GRAVITY_VARIANTS)
y_vars = sorted(cmp_table[COL_BETA_Y].unique())
y_pos = {v: i for i, v in enumerate(y_vars)}

fig = plt.figure(figsize=(FIG_WIDTH, BLOCK_HEIGHT * len(tags)))
outer = fig.add_gridspec(len(tags), 1, hspace=BLOCK_GAP)

for k, tag in enumerate(tags):
    country = VARIANTS[tag]["country"]
    cmp_c = cmp_table[cmp_table["country"] == country]

    # Two rows (histogram, beta) and one column per model inside the country block
    inner = outer[k].subgridspec(2, len(models), height_ratios=list(HEIGHT_RATIOS),
                                 hspace=ROW_GAP, wspace=COL_GAP)

    # Common x range inside a country block (b draws and beta intervals)
    b_all = boot.loc[boot["variant"] == tag, "b"].dropna()
    x_lo = min(b_all.min(), cmp_c[COL_BETA_LO].min())
    x_hi = max(b_all.max(), cmp_c[COL_BETA_HI].max())
    pad = X_PAD * (x_hi - x_lo)

    for j, model_name in enumerate(models):
        ax_h = fig.add_subplot(inner[0, j])
        ax_f = fig.add_subplot(inner[1, j], sharex=ax_h)

        sel = boot[(boot["variant"] == tag) & (boot["model"] == model_name)]
        b_valid = sel["b"].dropna()
        rows_cmp = cmp_c[cmp_c["model"] == model_name]
        b_main, b_lo, b_hi = rows_cmp.iloc[0][["b_main", "b_lo", "b_hi"]]

        # b interval and full-sample b run through both rows
        for ax in (ax_h, ax_f):
            ax.axvspan(b_lo, b_hi, color=CI_COLOR, alpha=0.6)
            ax.axvline(b_main, color="black", lw=2)

        # Top row: bootstrap distribution of b
        ax_h.hist(b_valid, bins=N_HIST_BINS, color=HIST_COLOR, alpha=0.85)
        ax_h.axvline(b_valid.median(), color="black", lw=1.5, ls="--")
        ax_h.text(0.02, 0.95, f"valid: {len(b_valid)}/{len(sel)}",
                  transform=ax_h.transAxes, va="top")
        ax_h.set_title(f"{country} - {MODEL_LABELS[model_name]}")
        plt.setp(ax_h.get_xticklabels(), visible=False)
        if j == 0:
            ax_h.set_ylabel(Y_LABEL)

        # Bottom row: scaling beta with its HC3 interval (red = no overlap with the b interval)
        for _, r in rows_cmp.iterrows():
            color = BETA_COLOR if r["ci_overlap"] else NO_OVERLAP_COLOR
            ax_f.errorbar(r[COL_BETA], y_pos[r[COL_BETA_Y]],
                          xerr=[[r[COL_BETA] - r[COL_BETA_LO]], [r[COL_BETA_HI] - r[COL_BETA]]],
                          fmt="o", color=color, capsize=3)
        ax_f.set_yticks(range(len(y_vars)))
        ax_f.set_yticklabels(y_vars if j == 0 else [])
        ax_f.set_ylim(len(y_vars) - 0.5, -0.5)
        ax_f.set_xlabel(X_LABEL)

        ax_h.set_xlim(x_lo - pad, x_hi + pad)        # shared with the beta row

# One shared legend under the last row
handles = [
    Patch(facecolor=CI_COLOR, alpha=0.6, label="b: 95% percentile interval"),
    Line2D([0], [0], color="black", lw=2, label="b (full-sample fit)"),
    Line2D([0], [0], color="black", lw=1.5, ls="--", label="b: bootstrap median"),
    Line2D([0], [0], marker="o", color=BETA_COLOR, label="beta with 95% HC3 interval"),
    Line2D([0], [0], marker="o", color=NO_OVERLAP_COLOR,
           label="beta interval does not overlap the b interval"),
]
fig.subplots_adjust(bottom=BOTTOM_MARGIN, top=0.95, left=0.2, right=0.97)
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)

fig.savefig(FIG_DIR / FIG_FILE, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("Saved:", os.path.relpath(FIG_DIR / FIG_FILE))

In [ ]:
# ----------------------------- CONFIG -----------------------------
CAPITAL_IDS = {"CZ": "CZ010", "PL": "PL911"}    # as in notebooks 05 and 05b
CAPITAL_COUNT_MAX = 2                           # draws with this many or more capitals are one group
CAPITAL_CSV = OUT_DIR / "capital_effect.csv"
CAPITAL_EXCLUDED_CSV = OUT_DIR / "capital_excluded_refit.csv"
# -------------------------------------------------------------------

bc = boot.copy()
bc["capital_count"] = np.nan

# 1) Regenerate every draw from its seed and count the capital in it
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    n_reg = len(ids)
    cap_pos = int(np.flatnonzero(ids == CAPITAL_IDS[c])[0])
    in_variant = bc["variant"] == tag

    cap_count, n_unique = {}, {}
    for i in bc.loc[in_variant, "draw"].unique():
        draw = np.random.default_rng(SEED_BASE + int(i)).integers(0, n_reg, size=n_reg)
        cap_count[i] = int((draw == cap_pos).sum())
        n_unique[i] = len(set(draw.tolist()))

    bc.loc[in_variant, "capital_count"] = bc.loc[in_variant, "draw"].map(cap_count)

    # The regenerated draws must match the saved ones (same number of unique regions)
    same = (bc.loc[in_variant, "draw"].map(n_unique) == bc.loc[in_variant, "n_unique_regions"]).all()
    print(f"{tag}: regenerated draws match the saved ones: {bool(same)}")

bc["capital_group"] = bc["capital_count"].clip(upper=CAPITAL_COUNT_MAX).astype(int)
by_capital = (bc.groupby(["variant", "model", "capital_group"])["b"]
              .agg(n_draws="size", n_valid="count", b_min="min",
                   b_q25=lambda s: s.quantile(0.25), b_median="median",
                   b_q75=lambda s: s.quantile(0.75), b_max="max")
              .reset_index())
by_capital.to_csv(CAPITAL_CSV, index=False)

# 2) Full-sample refit without the capital
rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    draw = np.flatnonzero(ids != CAPITAL_IDS[c])          # every region once, capital left out
    fdf_cap = build_fdf(c, draw, rule)
    saved = main_params[c].set_index("model")["destination_exp"]
    for model_name, r in fit_b(fdf_cap).items():
        rows.append({"variant": tag, "model": model_name, "n_regions": len(draw),
                     "n_pairs": len(fdf_cap), "b_with_capital": saved[model_name],
                     "b_without_capital": r["b"], "diff": r["b"] - saved[model_name],
                     "error_class": r["error_class"], "warnings": r["warnings"]})
excluded = pd.DataFrame(rows)
excluded.to_csv(CAPITAL_EXCLUDED_CSV, index=False)

print()
print(f"capital_group {CAPITAL_COUNT_MAX} means {CAPITAL_COUNT_MAX} or more copies of the capital")
print(by_capital.round(3).to_string(index=False))
print()
print(excluded.round(3).to_string(index=False))
print("Saved:", os.path.relpath(CAPITAL_CSV), "|", os.path.relpath(CAPITAL_EXCLUDED_CSV))

In [ ]:
import time

# ----------------------------- CONFIG -----------------------------
NOCAP_DIR = OUT_DIR / "boot_chunks_nocapital"
NOCAP_ALL_CSV = OUT_DIR / "boot_nocapital_all.csv"
NOCAP_CI_CSV = OUT_DIR / "b_nocapital_ci.csv"
# -------------------------------------------------------------------

NOCAP_DIR.mkdir(parents=True, exist_ok=True)

for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    non_cap_pos = np.flatnonzero(ids != CAPITAL_IDS[c])     # positions of the non-capital regions
    n_nc = len(non_cap_pos)

    for start in range(0, N_BOOT, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, N_BOOT)
        chunk_csv = NOCAP_DIR / f"bootnocap_{tag}_{start:04d}.csv"
        if chunk_csv.exists():
            continue                                        # resume: this chunk is already done

        t0 = time.time()
        rows = []
        for i in range(start, stop):
            rng = np.random.default_rng(SEED_BASE + i)
            draw = non_cap_pos[rng.integers(0, n_nc, size=n_nc)]   # non-capital regions, with replacement

            # A failed draw is recorded by class name, not raised
            try:
                fdf = build_fdf(c, draw, rule)
                n_pairs = len(fdf)
                res = fit_b(fdf)
            except Exception as e:
                n_pairs = np.nan
                res = {m: {"b": np.nan, "error_class": type(e).__name__, "warnings": ""}
                       for m in GRAVITY_VARIANTS}

            for model_name, r in res.items():
                rows.append({"variant": tag, "draw": i, "model": model_name,
                             "n_unique_regions": len(set(draw.tolist())), "n_pairs": n_pairs,
                             "b": r["b"], "error_class": r["error_class"],
                             "warnings": r["warnings"]})

        pd.DataFrame(rows).to_csv(chunk_csv, index=False)
        print(f"{tag} (no capital): draws {start}-{stop - 1} done in {time.time() - t0:.1f} s")

# Combine the chunks and summarise (percentile interval of the valid draws)
nocap = pd.concat([pd.read_csv(p) for tag in VARIANTS
                   for p in sorted(NOCAP_DIR.glob(f"bootnocap_{tag}_*.csv"))], ignore_index=True)
nocap["error_class"] = nocap["error_class"].fillna("")
nocap.to_csv(NOCAP_ALL_CSV, index=False)

q_lo, q_hi = (1 - CI_LEVEL) / 2, 1 - (1 - CI_LEVEL) / 2
rows = []
for (tag, model_name), g in nocap.groupby(["variant", "model"]):
    b_valid = g["b"].dropna()
    b_lo, b_hi = b_valid.quantile([q_lo, q_hi])
    b_nc = excluded.loc[(excluded["variant"] == tag) & (excluded["model"] == model_name),
                        "b_without_capital"].iloc[0]
    rows.append({"variant": tag, "model": model_name, "n_draws": len(g), "n_valid": len(b_valid),
                 "share_failed": 1 - len(b_valid) / len(g), "b_without_capital": b_nc,
                 "b_lo": b_lo, "b_median": b_valid.median(), "b_hi": b_hi,
                 "main_in_ci": bool(b_lo <= b_nc <= b_hi)})
nocap_ci = pd.DataFrame(rows)
nocap_ci.to_csv(NOCAP_CI_CSV, index=False)

print()
print(nocap_ci.round(4).to_string(index=False))
print("Saved:", os.path.relpath(NOCAP_ALL_CSV), "|", os.path.relpath(NOCAP_CI_CSV))

In [ ]:
# ----------------------------- CONFIG -----------------------------
BETA_SCALING_CSV = Path("../results/scaling_by_country/beta_by_country_2023.csv")
NOCAP_SAMPLE = "no_capitals"
NOCAP_CMP_CSV = OUT_DIR / "b_vs_beta_no_capital.csv"
# -------------------------------------------------------------------

beta_all = pd.read_csv(BETA_SCALING_CSV)
countries = [v["country"] for v in VARIANTS.values()]
beta_nc = beta_all[(beta_all["sample"] == NOCAP_SAMPLE) & (beta_all["group"].isin(countries))]
beta_nc = beta_nc[["group", "y_variable", "n_regions", "beta",
                   "beta_ci_low_hc3", "beta_ci_high_hc3"]].rename(columns={"group": "country"})

# Attach the country to the no-capital b table
b_nc = nocap_ci.copy()
b_nc["country"] = b_nc["variant"].map(lambda t: VARIANTS[t]["country"])

# One row per (country, model, y_variable)
cmp_nc = b_nc.merge(beta_nc, on="country", how="inner")
cmp_nc["diff_b_minus_beta"] = cmp_nc["b_without_capital"] - cmp_nc["beta"]
cmp_nc["ci_overlap"] = (cmp_nc["b_lo"] <= cmp_nc["beta_ci_high_hc3"]) & \
                       (cmp_nc["beta_ci_low_hc3"] <= cmp_nc["b_hi"])
cmp_nc["beta_in_b_ci"] = (cmp_nc["b_lo"] <= cmp_nc["beta"]) & (cmp_nc["beta"] <= cmp_nc["b_hi"])

cols = ["country", "model", "y_variable", "n_regions", "b_without_capital", "b_lo", "b_hi",
        "beta", "beta_ci_low_hc3", "beta_ci_high_hc3", "diff_b_minus_beta",
        "ci_overlap", "beta_in_b_ci"]
cmp_nc = cmp_nc[cols].sort_values(["country", "model", "y_variable"]).reset_index(drop=True)
cmp_nc.to_csv(NOCAP_CMP_CSV, index=False)

print(cmp_nc.round(3).to_string(index=False))
print("Saved:", os.path.relpath(NOCAP_CMP_CSV))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
FIG_NC_FILE = "b_vs_beta_no_capital.png"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b, power law (no capital)",
                "gravity_exp": "b, exponential (no capital)"}
BETA_COLOR = "#222222"
BAND_ALPHA = 0.12
PANEL_WIDTH, PANEL_HEIGHT = 5.6, 4.0
# -------------------------------------------------------------------

countries_nc = list(dict.fromkeys(cmp_nc["country"]))
fig, axes = plt.subplots(1, len(countries_nc),
                         figsize=(PANEL_WIDTH * len(countries_nc), PANEL_HEIGHT), squeeze=False)

for ax, c in zip(axes[0], countries_nc):
    sub = cmp_nc[cmp_nc["country"] == c]
    y_vars = sorted(sub["y_variable"].unique())
    y_pos = np.arange(len(y_vars))[::-1]                      # first variable on top

    # b without capital: value line + bootstrap interval band, per model
    for model_name, g in sub.groupby("model"):
        row = g.iloc[0]
        ax.axvspan(row["b_lo"], row["b_hi"], color=MODEL_COLORS[model_name], alpha=BAND_ALPHA, lw=0)
        ax.axvline(row["b_without_capital"], color=MODEL_COLORS[model_name], lw=1.8,
                   label=MODEL_LABELS[model_name])

    # beta with HC3 interval, one row per y variable (same beta for both models)
    beta_rows = sub.drop_duplicates("y_variable").set_index("y_variable").loc[y_vars]
    ax.errorbar(beta_rows["beta"], y_pos,
                xerr=[beta_rows["beta"] - beta_rows["beta_ci_low_hc3"],
                      beta_rows["beta_ci_high_hc3"] - beta_rows["beta"]],
                fmt="o", color=BETA_COLOR, capsize=3, label="beta (HC3 interval)")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_vars)
    ax.set_xlabel("exponent")
    ax.set_title(f"{c}: {int(sub['n_regions'].iloc[0])} regions, no capital")
    ax.grid(axis="x", alpha=0.3)

# One shared legend below the panels
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_NC_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
from src.scaling_stats import fit_scaling_law

# ----------------------------- CONFIG -----------------------------
MASTER_PARQUET = config.PROCESSED_DIR / "nuts3_master_table.parquet"
FIT_YEAR = 2023
POP_COL = "population"
MASTER_ID_COL = "nuts3_id"
MASTER_YEAR_COL = "year"
Y_COLS = ["gdp_total", "employment", "active_enterprises",
          "poi_count_university", "poi_count_office", "poi_count_library"]
BETA_BOOT_CSV = OUT_DIR / "boot_beta_all.csv"
SAVED_BETA_CSV = Path("../results/scaling_by_country/beta_by_country_2023.csv")
# -------------------------------------------------------------------

master_year = pd.read_parquet(MASTER_PARQUET)
master_year = master_year[master_year[MASTER_YEAR_COL] == FIT_YEAR].set_index(MASTER_ID_COL)

def beta_for_ids(draw_ids):
    """Fit the scaling law on the drawn regions (duplicates kept); return {y: (beta, n, error)}."""
    sample_df = master_year.loc[draw_ids]
    out = {}
    for y_col in Y_COLS:
        try:
            r = fit_scaling_law(sample_df, y_col=y_col, n_col=POP_COL)
            out[y_col] = (r["beta"], r["n_regions"], "")
        except Exception as e:
            out[y_col] = (np.nan, np.nan, type(e).__name__)   # class name only
    return out

if BETA_BOOT_CSV.exists():
    print("skip, already exists:", os.path.relpath(BETA_BOOT_CSV))
    beta_boot = pd.read_csv(BETA_BOOT_CSV)
else:
    rows = []
    for tag, rule in VARIANTS.items():
        c = rule["country"]
        ids = regions[c][COL_ID].to_numpy()
        assert set(ids) <= set(master_year.index), "region ids not found in master table"
        non_cap_pos = np.flatnonzero(ids != CAPITAL_IDS[c])
        samples = {"full": np.arange(len(ids)), "no_capitals": non_cap_pos}

        for sample, pool in samples.items():
            n_pool = len(pool)
            for i in range(N_BOOT):
                rng = np.random.default_rng(SEED_BASE + i)
                draw = pool[rng.integers(0, n_pool, size=n_pool)]   # same rule as the b bootstrap
                for y_col, (beta, n_reg, err) in beta_for_ids(ids[draw]).items():
                    rows.append({"variant": tag, "sample": sample, "draw": i, "y_variable": y_col,
                                 "beta": beta, "n_regions": n_reg, "error_class": err})
            print(tag, sample, "done")

    beta_boot = pd.DataFrame(rows)
    beta_boot.to_csv(BETA_BOOT_CSV, index=False)
    print("saved:", os.path.relpath(BETA_BOOT_CSV))

# Check: fit on the full original sample equals the saved beta of notebook 14
saved = pd.read_csv(SAVED_BETA_CSV)
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    for sample, sel in {"full": ids, "no_capitals": ids[ids != CAPITAL_IDS[c]]}.items():
        fit_now = beta_for_ids(sel)
        ref = saved[(saved["group"] == c) & (saved["sample"] == sample)].set_index("y_variable")["beta"]
        diff = max(abs(fit_now[y][0] - ref[y]) for y in Y_COLS)
        print(f"check {c} {sample}: max |beta - saved beta| = {diff:.2e}")

# Summary of the bootstrap betas (valid draws only)
summ = (beta_boot.dropna(subset=["beta"])
        .groupby(["variant", "sample", "y_variable"])["beta"]
        .agg(n_valid="count", beta_lo=lambda s: s.quantile(0.025),
             beta_median="median", beta_hi=lambda s: s.quantile(0.975)).reset_index())
print(summ.round(3).to_string(index=False))

In [ ]:
# ----------------------------- CONFIG -----------------------------
PAIRED_CSV = OUT_DIR / "paired_b_minus_beta.csv"
PAIRED_CI_LEVEL = 0.95
BOOT_SOURCES = {"full": BOOT_ALL_CSV, "no_capitals": NOCAP_ALL_CSV}   # b draws per sample
# -------------------------------------------------------------------

q_lo, q_hi = (1 - PAIRED_CI_LEVEL) / 2, 1 - (1 - PAIRED_CI_LEVEL) / 2

# Valid beta draws, keyed by variant, sample, draw
beta_valid = beta_boot.dropna(subset=["beta"])[["variant", "sample", "draw", "y_variable", "beta"]]

rows = []
for sample, path in BOOT_SOURCES.items():
    b_draws = pd.read_csv(path).dropna(subset=["b"])[["variant", "draw", "model", "b"]]
    merged = b_draws.merge(beta_valid[beta_valid["sample"] == sample],
                           on=["variant", "draw"], how="inner")
    merged["diff"] = merged["b"] - merged["beta"]

    for (tag, model_name, y_col), g in merged.groupby(["variant", "model", "y_variable"]):
        lo, hi = g["diff"].quantile([q_lo, q_hi])
        rows.append({"country": VARIANTS[tag]["country"], "sample": sample, "model": model_name,
                     "y_variable": y_col, "n_pairs": len(g),
                     "diff_median": g["diff"].median(), "diff_lo": lo, "diff_hi": hi,
                     "zero_in_ci": bool(lo <= 0 <= hi),
                     "spearman_b_beta": g["b"].corr(g["beta"], method="spearman")})

paired = (pd.DataFrame(rows)
          .sort_values(["country", "sample", "model", "y_variable"]).reset_index(drop=True))
paired.to_csv(PAIRED_CSV, index=False)
print(paired.round(3).to_string(index=False))
print("Saved:", os.path.relpath(PAIRED_CSV))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
FIG_PAIRED_FILE = "paired_b_minus_beta.png"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b, power law", "gravity_exp": "b, exponential"}
SAMPLE_ROWS = {"full": "full sample (capital included)", "no_capitals": "no capital"}
DODGE = 0.14          # vertical offset between the two models in a row
PANEL_WIDTH, PANEL_HEIGHT = 5.4, 3.6
# -------------------------------------------------------------------

countries_p = list(dict.fromkeys(paired["country"]))
fig, axes = plt.subplots(len(SAMPLE_ROWS), len(countries_p),
                         figsize=(PANEL_WIDTH * len(countries_p), PANEL_HEIGHT * len(SAMPLE_ROWS)),
                         sharex="col", squeeze=False)

offsets = {m: o for m, o in zip(MODEL_COLORS, (-DODGE, DODGE))}

for r, (sample, sample_label) in enumerate(SAMPLE_ROWS.items()):
    for k, c in enumerate(countries_p):
        ax = axes[r][k]
        sub = paired[(paired["country"] == c) & (paired["sample"] == sample)]
        y_vars = sorted(sub["y_variable"].unique())
        y_base = np.arange(len(y_vars))[::-1]                        # first variable on top

        for model_name, color in MODEL_COLORS.items():
            s = sub[sub["model"] == model_name].set_index("y_variable").loc[y_vars]
            ax.errorbar(s["diff_median"], y_base + offsets[model_name],
                        xerr=[s["diff_median"] - s["diff_lo"], s["diff_hi"] - s["diff_median"]],
                        fmt="o", color=color, capsize=2.5, markersize=5, lw=1.4,
                        label=MODEL_LABELS[model_name])

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(y_vars)
        ax.set_title(f"{c}: {sample_label}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(SAMPLE_ROWS) - 1:
            ax.set_xlabel("b - beta (median and 95% interval)")

# One shared legend below the panels
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_PAIRED_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
CO_Y_VARIABLE = "gdp_total"                  # change to another y variable to redraw
FIG_CO_FILE = f"b_beta_comodement_{CO_Y_VARIABLE}.png"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "power law", "gravity_exp": "exponential"}
SAMPLE_LABELS = {"full": "full sample", "no_capitals": "no capital"}
POINT_ALPHA = 0.25
PANEL_WIDTH, PANEL_HEIGHT = 3.6, 3.4
# -------------------------------------------------------------------

# Per-draw b and beta for the chosen y variable (same draws, valid in both)
panels = []                                   # (country, sample) in column order
merged_by_panel = {}
for sample, path in BOOT_SOURCES.items():
    b_draws = pd.read_csv(path).dropna(subset=["b"])[["variant", "draw", "model", "b"]]
    beta_y = beta_valid[(beta_valid["sample"] == sample) & (beta_valid["y_variable"] == CO_Y_VARIABLE)]
    m = b_draws.merge(beta_y[["variant", "draw", "beta"]], on=["variant", "draw"], how="inner")
    for tag, rule in VARIANTS.items():
        merged_by_panel[(rule["country"], sample)] = m[m["variant"] == tag]

col_keys = [(rule["country"], s) for rule in VARIANTS.values() for s in SAMPLE_LABELS]
fig, axes = plt.subplots(len(MODEL_COLORS), len(col_keys),
                         figsize=(PANEL_WIDTH * len(col_keys), PANEL_HEIGHT * len(MODEL_COLORS)),
                         squeeze=False)

for r, model_name in enumerate(MODEL_COLORS):
    for k, (c, sample) in enumerate(col_keys):
        ax = axes[r][k]
        g = merged_by_panel[(c, sample)]
        g = g[g["model"] == model_name]
        ax.scatter(g["beta"], g["b"], s=8, alpha=POINT_ALPHA, color=MODEL_COLORS[model_name], lw=0)

        # b = beta reference line over the shared range of the panel
        lo = min(g["beta"].min(), g["b"].min()); hi = max(g["beta"].max(), g["b"].max())
        ax.plot([lo, hi], [lo, hi], color="black", ls="--", lw=0.8)

        rho = g["b"].corr(g["beta"], method="spearman")
        ax.text(0.04, 0.94, f"Spearman = {rho:.2f}\nn = {len(g)}", transform=ax.transAxes,
                va="top", fontsize=9, bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
        if r == 0:
            ax.set_title(f"{c}: {SAMPLE_LABELS[sample]}", fontsize=10)
        if r == len(MODEL_COLORS) - 1:
            ax.set_xlabel(f"beta ({CO_Y_VARIABLE})")
        if k == 0:
            ax.set_ylabel(f"b, {MODEL_LABELS[model_name]}")
        ax.grid(alpha=0.25)

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_CO_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
FIG_SPEARMAN_FILE = "spearman_b_beta_overview.png"
MODEL_COLORS = {"gravity_power": "#0072B2", "gravity_exp": "#D55E00"}
MODEL_LABELS = {"gravity_power": "b, power law", "gravity_exp": "b, exponential"}
SAMPLE_ROWS = {"full": "full sample (capital included)", "no_capitals": "no capital"}
DODGE = 0.14
PANEL_WIDTH, PANEL_HEIGHT = 5.0, 3.4
# -------------------------------------------------------------------

countries_s = list(dict.fromkeys(paired["country"]))
fig, axes = plt.subplots(len(SAMPLE_ROWS), len(countries_s),
                         figsize=(PANEL_WIDTH * len(countries_s), PANEL_HEIGHT * len(SAMPLE_ROWS)),
                         sharex=True, squeeze=False)
offsets = {m: o for m, o in zip(MODEL_COLORS, (-DODGE, DODGE))}

for r, (sample, sample_label) in enumerate(SAMPLE_ROWS.items()):
    for k, c in enumerate(countries_s):
        ax = axes[r][k]
        sub = paired[(paired["country"] == c) & (paired["sample"] == sample)]
        y_vars = sorted(sub["y_variable"].unique())
        y_base = np.arange(len(y_vars))[::-1]                        # first variable on top

        for model_name, color in MODEL_COLORS.items():
            s = sub[sub["model"] == model_name].set_index("y_variable").loc[y_vars]
            ax.scatter(s["spearman_b_beta"], y_base + offsets[model_name], color=color, s=40,
                       label=MODEL_LABELS[model_name], zorder=3)

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(y_vars)
        ax.set_title(f"{c}: {sample_label}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(SAMPLE_ROWS) - 1:
            ax.set_xlabel("Spearman correlation of b and beta")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_SPEARMAN_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
# ----------------------------- CONFIG -----------------------------
STRATA_CSV = OUT_DIR / "spearman_by_capital_count.csv"
MIN_DRAWS = 30                          # minimum draws in a stratum to report a correlation
# CAPITAL_COUNT_MAX (cap on the copy count, "2+" group) comes from the capital cell
# -------------------------------------------------------------------

# Number of capital copies per draw (full-sample draw rule, same seeds as the bootstrap)
count_rows = []
for tag, rule in VARIANTS.items():
    c = rule["country"]
    ids = regions[c][COL_ID].to_numpy()
    for i in range(N_BOOT):
        rng = np.random.default_rng(SEED_BASE + i)
        draw = rng.integers(0, len(ids), size=len(ids))
        n_cap = int(np.sum(ids[draw] == CAPITAL_IDS[c]))
        count_rows.append({"variant": tag, "draw": i, "capital_copies": min(n_cap, CAPITAL_COUNT_MAX)})
counts = pd.DataFrame(count_rows)

b_full = pd.read_csv(BOOT_ALL_CSV).dropna(subset=["b"])[["variant", "draw", "model", "b"]]
beta_full = beta_valid[beta_valid["sample"] == "full"][["variant", "draw", "y_variable", "beta"]]
m = b_full.merge(beta_full, on=["variant", "draw"]).merge(counts, on=["variant", "draw"])

rows = []
for (tag, model_name, y_col, k), g in m.groupby(["variant", "model", "y_variable", "capital_copies"]):
    rho = g["b"].corr(g["beta"], method="spearman") if len(g) >= MIN_DRAWS else np.nan
    rows.append({"country": VARIANTS[tag]["country"], "model": model_name, "y_variable": y_col,
                 "capital_copies": k, "n_draws": len(g), "spearman_b_beta": rho})
strata = pd.DataFrame(rows)
strata.to_csv(STRATA_CSV, index=False)

# Compact view: one row per country, model, y variable; one column per stratum
view = strata.pivot_table(index=["country", "model", "y_variable"], columns="capital_copies",
                          values="spearman_b_beta")
view.columns = [f"rho_copies_{k}" for k in view.columns]
n_view = strata.pivot_table(index=["country", "model", "y_variable"], columns="capital_copies",
                            values="n_draws")
n_view.columns = [f"n_copies_{k}" for k in n_view.columns]
print(view.join(n_view).round(2).to_string())
print("Saved:", os.path.relpath(STRATA_CSV))

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------- CONFIG -----------------------------
FIG_STRATA_FILE = "spearman_by_capital_count.png"
MODEL_ROWS = {"gravity_power": "b, power law", "gravity_exp": "b, exponential"}
STRATUM_STYLE = {0: ("#009E73", "o", "0 capital copies"),      # green
                 1: ("#CC79A7", "s", "1 capital copy"),        # purple-pink
                 2: ("#000000", "^", "2+ capital copies")}     # black; key = capital_copies value
DODGE = 0.17
PANEL_WIDTH, PANEL_HEIGHT = 5.0, 3.4
# -------------------------------------------------------------------

countries_k = list(dict.fromkeys(strata["country"]))
fig, axes = plt.subplots(len(MODEL_ROWS), len(countries_k),
                         figsize=(PANEL_WIDTH * len(countries_k), PANEL_HEIGHT * len(MODEL_ROWS)),
                         sharex=True, squeeze=False)
offsets = {k: o for k, o in zip(STRATUM_STYLE, (-DODGE, 0, DODGE))}

for r, (model_name, model_label) in enumerate(MODEL_ROWS.items()):
    for j, c in enumerate(countries_k):
        ax = axes[r][j]
        sub = strata[(strata["country"] == c) & (strata["model"] == model_name)]
        y_vars = sorted(sub["y_variable"].unique())
        y_base = np.arange(len(y_vars))[::-1]                        # first variable on top

        for k, (color, marker, label) in STRATUM_STYLE.items():
            s = sub[sub["capital_copies"] == k].set_index("y_variable").reindex(y_vars)
            ax.scatter(s["spearman_b_beta"], y_base + offsets[k], color=color, marker=marker,
                       s=36, label=label, zorder=3)

        ax.axvline(0, color="black", ls="--", lw=1)
        ax.set_yticks(y_base)
        ax.set_yticklabels(y_vars)
        ax.set_title(f"{c}: {model_label}", fontsize=10)
        ax.grid(axis="x", alpha=0.3)
        if r == len(MODEL_ROWS) - 1:
            ax.set_xlabel("Spearman correlation of b and beta")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_path = FIG_DIR / FIG_STRATA_FILE
fig.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", os.path.relpath(fig_path))

In [ ]:
# ----------------------------- CONFIG -----------------------------
EQUIV_CSV = OUT_DIR / "equivalence_b_beta.csv"
EQUIV_LEVEL = 0.90                         # 90% interval = two one-sided tests at the 5% level
DELTAS = [0.1, 0.2, 0.3, 0.5]              # fixed margins shown for every case (not chosen after the fact)
# BOOT_SOURCES (sample -> b-draw csv) and beta_valid come from the paired-bootstrap cells
# -------------------------------------------------------------------

q_lo, q_hi = (1 - EQUIV_LEVEL) / 2, 1 - (1 - EQUIV_LEVEL) / 2

rows = []
for sample, path in BOOT_SOURCES.items():
    b_draws = pd.read_csv(path).dropna(subset=["b"])[["variant", "draw", "model", "b"]]
    merged = b_draws.merge(beta_valid[beta_valid["sample"] == sample],
                           on=["variant", "draw"], how="inner")
    merged["diff"] = merged["b"] - merged["beta"]

    for (tag, model_name, y_col), g in merged.groupby(["variant", "model", "y_variable"]):
        lo, hi = g["diff"].quantile([q_lo, q_hi])
        row = {"country": VARIANTS[tag]["country"], "sample": sample, "model": model_name,
               "y_variable": y_col, "n_pairs": len(g), "diff_lo90": lo, "diff_hi90": hi,
               "delta_min": max(abs(lo), abs(hi))}            # smallest margin with equivalence
        for d in DELTAS:
            row[f"equiv_delta_{d}"] = bool(-d <= lo and hi <= d)
        rows.append(row)

equiv = (pd.DataFrame(rows)
         .sort_values(["country", "sample", "model", "y_variable"]).reset_index(drop=True))
equiv.to_csv(EQUIV_CSV, index=False)

show = ["country", "sample", "model", "y_variable", "n_pairs", "delta_min"] + \
       [f"equiv_delta_{d}" for d in DELTAS]
print(equiv[show].round(2).to_string(index=False))
print("Saved:", os.path.relpath(EQUIV_CSV))